In [ ]:
%pip install kagglehub catboost xgboost tqdm -q
import kagglehub
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from xgboost import XGBClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier


# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
food_df = os.path.join(path, 'Q3_data.csv')
df_read = pd.read_csv(food_df)

print(f"Dataset shape: {df_read.shape}")
df_read.head()

In [ ]:
# Task 2: Write your code here:
df_read.head()

In [ ]:
# Task 3: Write your code here:
df_read.info()

In [ ]:
# Task 4: Write your code here:
df_read.describe()

In [ ]:
# Task 1: Write your code here:
def check_missing_values(df_read):
  missing_values = df_read.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_read)
#------------------------------------------------------------
categorical_cols = df_read.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

for col in df_read.columns:
    if df_read[col].dtype in ['float64', 'int64']:
        if df_read[col].isnull().any():
            median_val = df_read[col].median()
            df_read[col].fillna(median_val, inplace=True)

print("Missing values after imputation:")
print(df_read.isnull().sum()[df_read.isnull().sum() > 0])

if df_read.isnull().sum().sum() == 0:
    print("\nAll missing values have been handled.")
else:
    print("\nSome missing values remain (e.g., if there were non-numeric columns with NaNs that were not handled).")

In [ ]:
# Task 2: Write your code here:

def check_duplicates(df_read):
  duplicates = df_read.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_read.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_read)

In [ ]:
# Task 3: Write your code here:
categorical_cols = df_read.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))


In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df_read.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_read[numerical_cols] = scaler.fit_transform(df_read[numerical_cols])
df_read.head()

In [ ]:
# Task 5: Write your code here:

print("Target variable distribution:")
print(df_read['Target'].value_counts())
print("\nTarget variable percentages:")
print(df_read['Target'].value_counts(normalize=True) * 100)


if df_read['Target'].value_counts(normalize=True).min() < 0.2:
    print("\nThe target variable is imbalanced.")
else:
    print("\nThe target variable is not significantly imbalanced.")

In [ ]:
# Task 1: Write your code here:
X = df_read.drop("Target", axis=1).astype(float)
y = df_read['Target'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


f1_scores = []


for fold, (train_index, val_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold+1}/")

    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    model = CatBoostClassifier(random_state=42, verbose=0, iterations=100)
    model.fit(X_train, y_train)

    from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


f1_scores = []


for fold, (train_index, val_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold+1}/")

    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    model = CatBoostClassifier(random_state=42, verbose=0, iterations=100)
    model.fit(X_train, y_train)


    y_pred = model.predict(X_val)


    f1 = f1_score(y_val, y_pred)
    f1_scores.append(f1)
    print(f"F1-score for Fold {fold+1}: {f1:.4f}")


avg_f1_score = np.mean(f1_scores)
print(f"\nAverage F1-score across all folds: {avg_f1_score:.4f}")



In [ ]:
# Task 1: Write your code here:
feature_importances = model.get_feature_importance()
feature_names = X.columns


importance_series = pd.Series(feature_importances, index=feature_names)


sorted_importance = importance_series.sort_values(ascending=False)


top_n = 10
top_features = sorted_importance.head(top_n)


plt.figure(figsize=(12, 8))
top_features.plot(kind='barh')
plt.xlabel('Feature Importance')
plt.ylabel('Feature Name')
plt.title(f'Top {top_n} Most Important Features')
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# Task 2: Write your code here:
golden_feature = sorted_importance.index[0]
print(f"The golden feature (most important feature) is: {golden_feature}")

In [ ]:
# Task Bonus: Write your code here:
X = df_read